In [ ]:
!pip -q install -U pyserini transformers sentencepiece sacremoses
!apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

from google.colab import drive
drive.mount("/content/drive")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Mounted at /content/drive


In [ ]:
import json, textwrap
from pyserini.search.lucene import LuceneSearcher

class BM25Engine:
    def __init__(self, pack, lang="en", k1=0.9, b=0.4):
        self.lang = lang

        # docID -> (title, text), used to display results
        self.passages = {}
        with open(f"{pack}/passages.jsonl", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                self.passages[d["id"]] = (d["title"], d["text"])

        # BM25 index (the analyzer language must match the one used at indexing time)
        self.searcher = LuceneSearcher(f"{pack}/bm25_index")
        if lang != "en":
            self.searcher.set_language(lang)
        self.searcher.set_bm25(k1=k1, b=b)
        print(f"[{lang}] loaded: {self.searcher.num_docs:,} passages")

    def search(self, query, k=5):
        hits = self.searcher.search(query, k)
        results = []
        for rank, h in enumerate(hits, 1):
            title, text = self.passages[h.docid]
            results.append({"rank": rank, "docid": h.docid, "score": h.score,
                            "title": title, "text": text})
        return results

def show(results, query, extra=None, width=100, max_chars=500):
    print("=" * width)
    print(f"QUERY: {query}")
    if extra:
        print(extra)
    print("=" * width)
    if not results:
        print("No results.")
    for r in results:
        print(f"\n#{r['rank']}  score={r['score']:.3f}  docid={r['docid']}")
        print(f"Title: {r['title']}")
        body = r["text"][:max_chars] + ("..." if len(r["text"]) > max_chars else "")
        print(textwrap.fill(body, width=width, initial_indent="  ", subsequent_indent="  "))
    print("-" * width)

In [ ]:
en_engine = BM25Engine("/content/drive/MyDrive/miracl_en_indexes", lang="en")
hi_engine = BM25Engine("/content/drive/MyDrive/miracl_hi_indexes", lang="hi")

[en] loaded: 1,000,000 passages
[hi] loaded: 506,264 passages


In [ ]:
import os, json, time, math
import numpy as np, pandas as pd

BASE = "/content/drive/MyDrive/miracl_eval"
PACKS = {"en": "/content/drive/MyDrive/miracl_en_indexes",
         "hi": "/content/drive/MyDrive/miracl_hi_indexes"}
ENGINES = {"en": en_engine, "hi": hi_engine}

def load_set(lang, kind):
    """kind = 'eval' (all usable dev queries) or 'hard' (the 20-query suite)"""
    E = f"{BASE}/{lang}"
    if kind == "eval":
        q = json.load(open(f"{E}/eval_queries.json", encoding="utf-8"))
        r = json.load(open(f"{E}/eval_qrels.json"))
        meta = {}
    else:
        usable = set(json.load(open(f"{E}/hard_usable.json")))
        q = {k: v for k, v in json.load(open(f"{E}/hard_queries.json", encoding="utf-8")).items() if k in usable}
        r = json.load(open(f"{E}/hard_qrels.json"))
        meta = json.load(open(f"{E}/hard_meta.json"))
    return q, r, meta

def query_metrics(ranked, rel):
    """ranked: list of docids (top 100). rel: set of relevant docids. Binary relevance."""
    R = len(rel)
    p5   = sum(d in rel for d in ranked[:5]) / 5
    r100 = sum(d in rel for d in ranked[:100]) / R
    hits, s = 0, 0.0
    for i, d in enumerate(ranked[:100], 1):
        if d in rel:
            hits += 1
            s += hits / i
    ap   = s / R
    dcg  = sum(1 / math.log2(i + 1) for i, d in enumerate(ranked[:5], 1) if d in rel)
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(R, 5) + 1))
    return p5, r100, ap, dcg / idcg

def index_size_mb(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

def evaluate(engine, queries, qrels, label, idx_mb):
    engine.searcher.search(next(iter(queries.values())), 100)       # warm-up, not timed
    per_q, lat = {}, []
    for qid, text in queries.items():
        rel = {d for d, v in qrels[qid].items() if v > 0}
        t = time.perf_counter()
        hits = engine.searcher.search(text, 100)                    # pure retrieval time
        lat.append((time.perf_counter() - t) * 1000)
        per_q[qid] = query_metrics([h.docid for h in hits], rel)
    m = np.mean(list(per_q.values()), axis=0)
    row = {"Model": label, "n_queries": len(queries),
           "P@5": m[0], "R@100": m[1], "MAP": m[2], "NDCG@5": m[3],
           "Latency (ms)": np.mean(lat), "Index (MB)": idx_mb}
    return row, per_q

In [ ]:
rows, per_query_all = [], {}
for lang in ["en", "hi"]:
    idx_mb = index_size_mb(f"{PACKS[lang]}/bm25_index")
    for kind in ["eval", "hard"]:
        if not os.path.exists(f"{BASE}/{lang}/{'eval_queries' if kind=='eval' else 'hard_usable'}.json"):
            print(f"skipping {lang}/{kind}: file missing"); continue
        q, r, meta = load_set(lang, kind)
        row, per_q = evaluate(ENGINES[lang], q, r, f"BM25 {lang}→{lang} [{kind}]", idx_mb)
        rows.append(row); per_query_all[(lang, kind)] = per_q

        # per-category breakdown for the hard suite
        if kind == "hard":
            for cat in sorted(set(meta.values())):
                ids = [x for x in per_q if meta[x] == cat]
                m = np.mean([per_q[x] for x in ids], axis=0)
                rows.append({"Model": f"   └ {cat}", "n_queries": len(ids),
                             "P@5": m[0], "R@100": m[1], "MAP": m[2], "NDCG@5": m[3]})

df = pd.DataFrame(rows)
pd.set_option("display.width", 200)
display(df.round(3))

os.makedirs(f"{BASE}/results", exist_ok=True)
df.to_csv(f"{BASE}/results/bm25_results.csv", index=False)

,Model,n_queries,P@5,R@100,MAP,NDCG@5,Latency (ms),Index (MB)
0,BM25 en→en [eval],230,0.174,0.908,0.436,0.469,111.785,791.901
1,BM25 en→en [hard],9,0.178,1.000,0.362,0.371,13.694,791.901
2,└ homonym_ambiguity,3,0.000,1.000,0.111,0.000,NaN,NaN
3,└ negation,3,0.267,1.000,0.567,0.679,NaN,NaN
4,└ temporal,3,0.267,1.000,0.408,0.435,NaN,NaN
5,BM25 hi→hi [eval],350,0.199,0.868,0.400,0.415,22.886,46.575
6,BM25 hi→hi [hard],20,0.270,0.812,0.453,0.467,28.335,46.575
7,└ homonym_ambiguity,7,0.229,0.821,0.473,0.465,NaN,NaN
8,└ negation,7,0.229,0.714,0.408,0.425,NaN,NaN
9,└ temporal,6,0.367,0.917,0.481,0.518,NaN,NaN


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_NAME = "Helsinki-NLP/opus-mt-hi-en"
device = "cuda" if torch.cuda.is_available() else "cpu"
mt_tok = AutoTokenizer.from_pretrained(MT_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(device).eval()

def translate_hi_en(text):
    inputs = mt_tok(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = mt_model.generate(**inputs, max_new_tokens=128, num_beams=4)
    return mt_tok.decode(out[0], skip_special_tokens=True)

# eyeball the translation quality on a few hard Hindi queries
hq, _, hmeta = load_set("hi", "hard")
for qid, text in list(hq.items())[:8]:
    print(f"[{hmeta[qid]}] {text}\n   → {translate_hi_en(text)}\n")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[negation] जैन धर्म के लोग प्याज क्यों नही खाते?
   → Why is Jain's religious background not popular?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[negation] यूरोप के अलावा, कौन सा एकमात्र महाद्वीप है जिसमें से देशों ने पुरुषों के लिए फुटबॉल विश्व कप जीता है?
   → Besides Europe, what is the only continent from which countries have won football world cup for men?

[negation] घर्षण द्वारा किए गए कार्य का क्षय नहीं होता है बल्कि यह ऊर्जा में रूपांतरित हो जाती है?
   → It does not decay the work done by blocking but is transformed into energy?



...

[negation] भारत के राष्ट्रवादी नेताओं ने साइमन कमीशन का बहिष्कार किया था, क्योंकि?
   → The national leaders of India had disfellowshipped Simon Parliament, because?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[negation] कितनी बार एक वर्ष में कम से कम संसद की बैठक होना आवश्यक है?
   → How often must there be a meeting of the Parliament at least one year?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[negation] किस राज्य ने राज्य में शराब का सेवन पूरी बंद कर दिया है?
   → What Kingdom has totally stopped the use of alcohol in the Kingdom?

[homonym_ambiguity] क्रिकेट नाम किस प्रसिद्ध ब्रिटिश व्यक्ति ने दिया था?
   → What famous British man named cricket?

